In [ ]:
from pathlib import Path
import pandas
import numpy as np

def process_csv(input_path, output_path):
    df = pandas.read_csv(input_path)
    df = df.drop('ts', axis=1)
    df["Time (s)"] = df["t_s"].round(2)
    features_df = (df.pivot_table(
            index="Time (s)",
            columns="counter_name",
            values="value",
            aggfunc="mean", # if multiple rows have same counter_name and time bin, mean value is used
            observed=True)
          .sort_index())
    features_df = features_df.reset_index()
    features_df.columns.name = None
    features_df = features_df.replace(0, np.nan)
    features_df = features_df.dropna(axis=1, how='all')
    features_df = features_df.ffill()
    features_df = features_df.bfill()
    features_df = features_df.dropna(axis=1, how='any')
    features_df = features_df[features_df['Time (s)'] >= 7]
    if features_df.shape[1] >= 50:
        features_df.to_csv(output_path, index=False)

to_clean = Path("meta_scan_csvs/uncleaned/pre_update")
cleaned_base = Path("meta_scan_csvs/cleaned")
for csv_file in to_clean.rglob("*.csv"):
    relative_path = csv_file.relative_to(to_clean)
    output_file = cleaned_base / relative_path.parent / f"{csv_file.stem}_cleaned.csv"
    print(f"Processing {csv_file.relative_to(to_clean)}...")
    output_file.parent.mkdir(parents=True, exist_ok=True)
    process_csv(csv_file, output_file)
    print(f"Created {output_file.name}\n")